# Ddri 군집화 전처리 설명 노트북

## 목적

- 강남구 따릉이 군집화 전에 어떤 전처리 기준을 적용했는지 설명한다.
- 결측치, 이상치, 범위 필터링 기준을 재현 가능한 형태로 정리한다.
- 전처리 결과가 이후 군집화 feature에 어떤 영향을 주는지 확인한다.

## 분석 범위

- 공간 범위: 강남구 공통 운영 대여소
- 학습 기준 기간: 2023-01-01 ~ 2024-12-31
- 테스트 기준 기간: 2025-01-01 ~ 2025-12-31

## 입력 데이터

- 강남구 따릉이 이용정보 CSV
- 강남구 대여소 마스터 CSV
- 공통 대여소 기준 정보

## 실제 적용 전처리 기준

- 필수 컬럼 결측치 제거
- `이용시간(분) <= 0` 제거
- `이용거리(M) <= 0` 제거
- 공통 기준 밖 대여소 제거
- 강남구 기준 밖 반납 대여소 제거
- 동일 대여소 반납은 제거하지 않고 유지

## 해석 시 주의점

- 동일 대여소 반납은 이상치가 아니라 운영 해석용 보조 지표로 본다.
- 극단치 후보는 존재하지만 1차 군집화에서는 자동 제거하지 않았다.
- 이 노트북은 설명용이며, 실제 재생성 기준 코드는 `ddri_report_chart_builder.py`, `ddri_station_clustering_baseline.py`를 따른다.


## 1. 기본 설정과 입력 경로

이 단계에서는 프로젝트 루트와 전처리 결과 파일 경로를 불러온다. 이후 셀에서는 이미 생성된 로그와 요약 파일을 읽어 설명 중심으로 확인한다.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
PREP_DIR = BASE_DIR / 'works' / '01_clustering' / '02_preprocessing'
DATA_DIR = PREP_DIR / 'data'
IMG_DIR = PREP_DIR / 'images'

cleaning_log_path = DATA_DIR / 'ddri_cleaning_log.csv'
cleaning_summary_path = DATA_DIR / 'ddri_cleaning_summary_by_group.csv'
dup_summary_path = DATA_DIR / 'ddri_duplicate_check_summary.csv'
iqr_summary_path = DATA_DIR / 'ddri_feature_iqr_outlier_summary.csv'

cleaning_log_path, cleaning_summary_path

(PosixPath('/Users/cheng80/Desktop/ddri_work/works/01_clustering/02_preprocessing/data/ddri_cleaning_log.csv'),
 PosixPath('/Users/cheng80/Desktop/ddri_work/works/01_clustering/02_preprocessing/data/ddri_cleaning_summary_by_group.csv'))

## 2. 전처리 로그 확인

전처리 로그에는 파일별 입력 행 수, 제거 후 행 수, 제거 사유가 기록되어 있다. 이 표를 통해 어떤 필터가 실제로 크게 작동했는지 확인할 수 있다.

In [2]:
cleaning_log = pd.read_csv(cleaning_log_path)
cleaning_log.head()

,file_name,rows_before,rows_after,dropped_missing,dropped_nonpositive,dropped_noncommon_rent,dropped_outside_gangnam_return,group_name
0,2301_강남구_따릉이_이용정보.csv,34937,31247,0,2778,912,0,train_2023
1,2302_강남구_따릉이_이용정보.csv,49372,44539,0,3480,1353,0,train_2023
2,2303_강남구_따릉이_이용정보.csv,80572,72861,0,5431,2280,0,train_2023
3,2304_강남구_따릉이_이용정보.csv,91717,82939,0,6235,2543,0,train_2023
4,2305_강남구_따릉이_이용정보.csv,107880,98133,0,6786,2961,0,train_2023


## 3. 그룹별 요약 확인

학습/테스트 기준으로 제거 규모를 요약해서 보면, 비정상 시간·거리 값과 기준 밖 대여소 제거 영향이 컸다는 점을 확인할 수 있다.

In [3]:
cleaning_summary = pd.read_csv(cleaning_summary_path)
cleaning_summary

,group_name,rows_before,rows_after,dropped_missing,dropped_nonpositive,dropped_noncommon_rent,dropped_outside_gangnam_return
0,test_2025,869397,825111,0,15022,29264,0
1,train_2023,1012480,917202,0,68978,26300,0
2,train_2024,1014918,943381,0,54057,17480,0


## 4. 중복 및 극단치 점검

중복 로그와 feature 극단치 후보는 별도로 점검했다. 이 단계는 무조건 삭제하기 위한 것이 아니라, 왜 유지 또는 제거했는지 설명하기 위한 근거를 만드는 과정이다.

In [4]:
dup_summary = pd.read_csv(dup_summary_path)
iqr_summary = pd.read_csv(iqr_summary_path)

display(dup_summary)
display(iqr_summary.head())

,files,rows,dup_all,dup_key
0,36,2896795,21,21


,feature,q1,q3,iqr,lower_bound,upper_bound,outlier_count
0,avg_rental,8.656448,19.009784,10.353336,-6.873557,34.539788,12
1,rental_std,5.547695,11.063102,5.515407,-2.725415,19.336213,6
2,weekday_avg,9.433483,20.984080,11.550597,-7.892412,38.309975,11
3,weekend_avg,6.867890,15.705529,8.837638,-6.388567,28.961986,13
4,peak_ratio,0.338877,0.403111,0.064234,0.242525,0.499463,6


## 5. 차트 경로와 발표 활용

전처리 설명에 직접 사용할 수 있는 차트는 아래 두 장이다.

- `ddri_cleaning_before_after.png`
- `ddri_cleaning_drop_reasons.png`

이 차트들은 발표와 레포트에서 전처리 근거를 직관적으로 보여주는 역할을 한다.